In [1]:
# Create a new cell with this code:

# First uninstall conflicting packages
!pip uninstall -y numpy thinc

# Install the required versions in the right order
!pip install numpy==1.26.0  # Installing a specific version that should work with thinc
!pip install sagemaker boto3

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check if it's mounted properly
!ls /content/drive/MyDrive

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: thinc 8.3.6
Uninstalling thinc-8.3.6:
  Successfully uninstalled thinc-8.3.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 87.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.5 requires thinc<8.4.0,>=8.3.4, which is not installed.
sagemaker 2.244.0 requires numpy==1.26.4, but you have numpy 1.26.0 which is incompatible.
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.0
    Uninstalling nu

In [ ]:
# Import necessary libraries
import sagemaker
import boto3
from sklearn.model_selection import train_test_split
import pandas as pd
import os

# Set up AWS credentials and region explicitly
# Replace with your NEW credentials after rotating the exposed ones
os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

# Now set up AWS/Sagemaker connection with explicit region
boto3_sm = boto3.client(
    "sagemaker",
    region_name="us-east-1",
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

# Create the session with the client
session = sagemaker.Session(boto_session=boto3.Session(
    region_name="us-east-1",
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
))

region = session.boto_session.region_name
bucket = "crewasis-happiness2020"
print("Using Bucket:", bucket)

# Read the CSV file from your Google Drive
df = pd.read_csv("/content/drive/MyDrive/mpc_train.csv")

# Display the shape to confirm data is loaded
print(df.shape)

# Let's also see a preview of the data
print("\nPreview of the data:")
print(df.head())

Using Bucket: crewasis-happiness2020
(2000, 21)

Preview of the data:
   battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \
0        188        2  ...         20       756  2549     9     7         19   
1        136        3  ...        905      1988  2631    17     3          7   
2        145        5  ...       1263      1716  2603    11     2          9   
3        131        6  ...       1216      1786  2769    16     8         11   
4        141        2  ...       1208      1212  

In [8]:
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [9]:
df["price_range"].value_counts()

,count
price_range,
1,500
2,500
3,500
0,500


In [10]:
df.columns

Index(['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g',
       'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height',
       'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g',
       'touch_screen', 'wifi', 'price_range'],
      dtype='object')

In [11]:
# Percentage of values that are missing
df.isnull().sum()*100

,0
battery_power,0
blue,0
clock_speed,0
dual_sim,0
fc,0
four_g,0
int_memory,0
m_dep,0
mobile_wt,0
n_cores,0


In [12]:
features = list(df.columns)
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi',
 'price_range']

In [13]:
label = features.pop(-1)
label

'price_range'

In [14]:
x = df[features]
y = df[label]

In [15]:
y.head()

,price_range
0,1
1,2
2,2
3,2
4,1


In [16]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [17]:
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X_train Shape: (1600, 20)
X_test Shape: (400, 20)
y_train Shape: (1600,)
y_test Shape: (400,)


In [18]:
trainX = pd.DataFrame(X_train)
trainX['label'] = y_train

testX = pd.DataFrame(X_test)
testX['label'] = y_test

In [19]:
print(trainX.shape)
print(testX.shape)

(1600, 21)
(400, 21)


In [20]:
trainX.to_csv('train_v1.csv', index=False)
testX.to_csv('test_v1.csv', index=False)

In [21]:
# Sagemaker takes training data from S3 bucket, so uploading data to S3 bucket
sk_folder = "sagemaker/mobile_price_classification/sklearncontainer"

train_path = session.upload_data(path="train_v1.csv", bucket=bucket, key_prefix=sk_folder)
test_path = session.upload_data(path="test_v1.csv", bucket=bucket, key_prefix=sk_folder)

In [22]:
print(train_path)

s3://crewasis-happiness2020/sagemaker/mobile_price_classification/sklearncontainer/train_v1.csv


In [23]:
%%writefile script.py

import argparse
import os
import json
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
import joblib
import pathlib
from io import StringIO
import boto3
import pandas as pd
import numpy as np

def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf

if __name__ =='__main__':

    print("[INFO] Extracting arguments")
    parser = argparse.ArgumentParser()

    # hyperparameters sent by the client are passed as command-line arguments to the script.
    parser.add_argument('--n_estimators', type=int, default=100)
    parser.add_argument('--random_state', type=int, default=0)
    # parser.add_argument('--epochs', type=int, default=10)
    # parser.add_argument('--batch-size', type=int, default=100)
    # parser.add_argument('--learning-rate', type=float, default=0.1)

    # an alternative way to load hyperparameters via SM_HPS environment variable.
    # parser.add_argument('--sm-hps', type=json.loads, default=os.environ['SM_HPS'])

    # input data and model directories
    parser.add_argument('--model-dir', type=str, default=os.environ['SM_MODEL_DIR'])
    parser.add_argument('--train', type=str, default=os.environ['SM_CHANNEL_TRAIN'])
    parser.add_argument('--test', type=str, default=os.environ['SM_CHANNEL_TEST'])
    parser.add_argument('--train_file', type=str, default='train_v1.csv')
    parser.add_argument('--test_file', type=str, default='test_v1.csv')

    args, _ = parser.parse_known_args()

    print("SKLearn Version: ", sklearn.__version__)
    print("Joblib Version: ", joblib.__version__)

    print("[INFO] Reading data")
    print()
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing datasets")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print('Column order: ')
    print(features)
    print()

    print("Label column is: ",label)
    print()

    print("Data Shape: ")
    print()
    print("---- SHAPE OF TRAINING DATA (80%) ----")
    print(X_train.shape)
    print(y_train.shape)
    print()
    print("---- SHAPE OF TESTING DATA (20%) ----")
    print(X_test.shape)
    print(y_test.shape)
    print()

    print("Training RandomForest Model.....")
    print()
    model = RandomForestClassifier(n_estimators=args.n_estimators, random_state=args.random_state, verbose=3, n_jobs=None)
    model.fit(X_train, y_train)
    print()

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)
    print("Model persisted at " + model_path)
    print()


    y_pred_test = model.predict(X_test)
    test_acc = accuracy_score(y_test,y_pred_test)
    test_rep = classification_report(y_test,y_pred_test)

    print()
    print("---- METRICS RESULTS FOR TESTING DATA ----")
    print()
    print("Total Rows are: ", X_test.shape[0])
    print('[TESTING] Model Accuracy is: ', test_acc)
    print('[TESTING] Testing Report: ')
    print(test_rep)

Writing script.py


In [28]:
from sagemaker.sklearn.estimator import SKLearn

FRAMEWORK_VERSION = "0.23-1"

sklearn_estimator = SKLearn(
    entry_point = "script.py",
    role="arn:aws:iam::851725210627:role/service-role/AmazonSageMaker-ExecutionRole-20250207T013491",
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": 100,
        "random_state": 0,
    },
    use_spot_instances = True,
    max_wait = 7200,
    max_run = 3600
)

In [29]:
sklearn_estimator.fit({"train":train_path, "test":test_path}, wait=True)

2025-05-03 16:54:34 Starting - Starting the training job...
2025-05-03 16:55:08 Downloading - Downloading input data...
2025-05-03 16:55:34 Downloading - Downloading the training image...
2025-05-03 16:56:14 Training - Training image download completed. Training in progress..2025-05-03 16:56:19,135 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-05-03 16:56:19,137 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-05-03 16:56:19,176 sagemaker_sklearn_container.training INFO     Invoking user training script.
2025-05-03 16:56:19,341 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-05-03 16:56:19,353 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-05-03 16:56:19,365 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-05-03 16:56:19,373 sagemaker-training-toolkit INFO     Invoking user script
Trai

In [30]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = boto3_sm.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact present at: ", artifact)



2025-05-03 16:56:37 Starting - Preparing the instances for training
2025-05-03 16:56:37 Downloading - Downloading the training image
2025-05-03 16:56:37 Training - Training image download completed. Training in progress.
2025-05-03 16:56:37 Uploading - Uploading generated training model
2025-05-03 16:56:37 Completed - Training job completed
Model artifact present at:  s3://sagemaker-us-east-1-851725210627/RF-custom-sklearn-2025-05-03-16-54-29-196/output/model.tar.gz


In [31]:
artifact

's3://sagemaker-us-east-1-851725210627/RF-custom-sklearn-2025-05-03-16-54-29-196/output/model.tar.gz'

In [32]:
# Making a copy of the built model for deployment
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name = "Custom-sklearn-model" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role="arn:aws:iam::851725210627:role/service-role/AmazonSageMaker-ExecutionRole-20250207T013491",
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION
)

In [33]:
model

In [34]:
# Endpoint Deployment
endpoint_name = "Custom-sklearn-model" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName={}".format(endpoint_name))

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name
)

EndpointName=Custom-sklearn-model2025-05-03-17-14-11
--------!

In [35]:
endpoint_name

'Custom-sklearn-model2025-05-03-17-14-11'

In [36]:
# First five rows of the test dataset for prediction using the endpoint created above
testX[features][0:5].values.tolist()

[[1454.0,
  1.0,
  0.5,
  1.0,
  1.0,
  0.0,
  34.0,
  0.7,
  83.0,
  4.0,
  3.0,
  250.0,
  1033.0,
  3419.0,
  7.0,
  5.0,
  5.0,
  1.0,
  1.0,
  0.0],
 [1092.0,
  1.0,
  0.5,
  1.0,
  10.0,
  0.0,
  11.0,
  0.5,
  167.0,
  3.0,
  14.0,
  468.0,
  571.0,
  737.0,
  14.0,
  4.0,
  11.0,
  0.0,
  1.0,
  0.0],
 [1524.0,
  1.0,
  1.8,
  1.0,
  0.0,
  0.0,
  10.0,
  0.6,
  174.0,
  4.0,
  1.0,
  154.0,
  550.0,
  2678.0,
  16.0,
  5.0,
  13.0,
  1.0,
  0.0,
  1.0],
 [1807.0,
  1.0,
  2.1,
  0.0,
  2.0,
  0.0,
  49.0,
  0.8,
  125.0,
  1.0,
  10.0,
  337.0,
  1384.0,
  1906.0,
  17.0,
  13.0,
  13.0,
  0.0,
  1.0,
  1.0],
 [1086.0,
  1.0,
  1.7,
  1.0,
  0.0,
  1.0,
  43.0,
  0.2,
  111.0,
  6.0,
  1.0,
  56.0,
  1150.0,
  3285.0,
  11.0,
  5.0,
  17.0,
  1.0,
  1.0,
  0.0]]

In [37]:
print(predictor.predict(testX[features][0:5].values.tolist()))

[3 0 2 1 3]


In [38]:
# Deleting the endpoint to avoid the costs
boto3_sm.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': '2fad23f2-3934-4415-847d-bca16d8f2c77',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2fad23f2-3934-4415-847d-bca16d8f2c77',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Sat, 03 May 2025 17:41:19 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}